# ⚽ Prophetia2: Dashboard Interactivo de Predicciones

Este dashboard simula una sala de análisis en tiempo real. Utilizaremos **ipywidgets** para seleccionar partidos del conjunto de datos y ver la **predicción de nuestro modelo** junto con la comparación visual de estadísticas clave.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Estilo general
plt.style.use('dark_background')
sns.set_theme(style="darkgrid", palette="pastel")

def cargar_recursos():
    try:
        df = pd.read_parquet('../data/processed/matches_dataset.parquet')
        saved_data = joblib.load('../core/save_models/prophetia_xgb_model.pkl')
        return df, saved_data['model'], saved_data['features']
    except Exception as e:
        print(f"Error cargando recursos: {e}")
        return None, None, None

df, modelo, feature_cols = cargar_recursos()

## Visualizador de Partidos

In [ ]:
if df is not None and modelo is not None and feature_cols is not None:
    # Limpiar posibles nulos
    df = df.fillna(0)
    
    # Agrupar partidos para el selector
    match_list = df[df['is_home'] == 1].copy()
    
    opciones = []
    for idx, row in match_list.iterrows():
        year = row['match_date'][:4] if isinstance(row['match_date'], str) else str(row['match_date'].year)
        texto = f"{year} - {row['team']} vs {row['opponent']}"
        opciones.append((texto, row['match_id']))
        
    dropdown_matches = widgets.Dropdown(
        options=opciones,
        description='Partido:',
        style={'description_width': 'initial'},
        layout={'width': '500px'}
    )

    output_area = widgets.Output()

    def actualizar_dashboard(cambio):
        with output_area:
            clear_output(wait=True)
            match_id = dropdown_matches.value
            
            partido = df[df['match_id'] == match_id].copy()
            
            if len(partido) != 2:
                print("Error en los datos del partido.")
                return
                
            home_row = partido[partido['is_home'] == 1].iloc[0]
            away_row = partido[partido['is_home'] == 0].iloc[0]
            
            home_team = home_row['team']
            away_team = away_row['team']
            
            # Preparar la data para predicción usando las variables exactas con las que se entrenó
            X_home_full = home_row[feature_cols].to_frame().T
            
            # Predecir probabilidades (el pipeline hace todo el feature selection y escalado internamente)
            probs = modelo.predict_proba(X_home_full)[0]
            clases = modelo.classes_
            
            prob_loss = probs[np.where(clases == 0)[0][0]] * 100
            prob_draw = probs[np.where(clases == 1)[0][0]] * 100
            prob_win = probs[np.where(clases == 2)[0][0]] * 100
            
            html_header = f"""
            <div style='background-color:#1e272e; padding:20px; border-radius:10px; color:white; text-align:center;'>
                <h2>⚽ {home_team} vs {away_team}</h2>
                <h3>Resultado Real: {home_row['goals_scored']} - {away_row['goals_scored']}</h3>
                <hr style='border-color:#485460;'>
                <div style='display:flex; justify-content:space-around;'>
                    <div>
                        <h4 style='color:#2ecc71;'>Probabilidad Victoria ({home_team})</h4>
                        <h2>{prob_win:.1f}%</h2>
                    </div>
                    <div>
                        <h4 style='color:#f1c40f;'>Probabilidad Empate</h4>
                        <h2>{prob_draw:.1f}%</h2>
                    </div>
                    <div>
                        <h4 style='color:#e74c3c;'>Probabilidad Victoria ({away_team})</h4>
                        <h2>{prob_loss:.1f}%</h2>
                    </div>
                </div>
            </div>
            """
            display(HTML(html_header))
            
            # Gráficos Comparativos
            metrics_to_plot = ['xg_created', 'shots_on_target', 'corners', 'passes_completed', 'pressures']
            
            fig, ax = plt.subplots(figsize=(10, 5))
            x = np.arange(len(metrics_to_plot))
            width = 0.35
            
            home_vals = [home_row[m] for m in metrics_to_plot]
            away_vals = [away_row[m] for m in metrics_to_plot]
            
            rects1 = ax.bar(x - width/2, home_vals, width, label=home_team, color='#3498db')
            rects2 = ax.bar(x + width/2, away_vals, width, label=away_team, color='#e74c3c')
            
            ax.set_ylabel('Valor')
            ax.set_title('Comparación de Estadísticas Clave')
            ax.set_xticks(x)
            ax.set_xticklabels(['xG', 'Tiros a Puerta', 'Córners', 'Pases Compl.', 'Presiones'])
            ax.legend()
            
            ax.bar_label(rects1, fmt='%.1f', padding=3, color='white')
            ax.bar_label(rects2, fmt='%.1f', padding=3, color='white')
            
            plt.tight_layout()
            plt.show()

    dropdown_matches.observe(actualizar_dashboard, names='value')
    display(dropdown_matches)
    display(output_area)
    actualizar_dashboard(None)
else:
    print("Por favor asegúrate de haber ejecutado 'core/data_adapter.py', 'core/feature_engineering.py' y 'core/train.py' antes de usar el dashboard.")